# Insert ELI Test Data

In [ ]:
import requests
import json
import uuid
import time

# Virtuoso SPARQL endpoint (direct, bypassing mu-auth)
SPARQL_URL = "http://localhost:8890/sparql"

# Graphs used by the stack
PUBLIC_GRAPH = "http://mu.semte.ch/graphs/public"
PDF_GRAPH = "http://mu.semte.ch/graphs/public/pdf"
HARVESTING_GRAPH = "http://mu.semte.ch/graphs/harvesting"

PREFIXES = """
PREFIX eli:   <http://data.europa.eu/eli/ontology#>
PREFIX epvoc: <https://data.europarl.europa.eu/def/epvoc#>
PREFIX dct:   <http://purl.org/dc/terms/>
PREFIX mu:    <http://mu.semte.ch/vocabularies/core/>
PREFIX skos:  <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>
PREFIX prov:  <http://www.w3.org/ns/prov#>
PREFIX oa:    <http://www.w3.org/ns/oa#>
PREFIX foaf:  <http://xmlns.com/foaf/0.1/>
PREFIX org:   <http://www.w3.org/ns/org#>
PREFIX mjp:   <http://data.lblod.info/vocabularies/mjp/>
"""


def sparql_update(query: str):
    resp = requests.post(SPARQL_URL, data={"query": query})
    resp.raise_for_status()
    return resp


def sparql_query(query: str) -> list[dict]:
    resp = requests.get(SPARQL_URL, params={"query": query, "format": "application/json"})
    resp.raise_for_status()
    return resp.json().get("results", {}).get("bindings", [])


print("Setup complete.")

## 1. Health Check - Verify Virtuoso is accessible

In [ ]:
try:
    result = sparql_query("SELECT (COUNT(*) as ?count) WHERE { ?s ?p ?o }")
    count = result[0]["count"]["value"] if result else "0"
    print(f"Virtuoso is accessible. Total triples: {count}")
except requests.exceptions.ConnectionError:
    print("ERROR: Cannot connect to Virtuoso at port 8890.")
    print("Make sure the stack is running and Virtuoso port is published.")
    print("You may need to add to docker-compose.override.yml:")
    print('  virtuoso:')
    print('    ports:')
    print('      - "8890:8890"')

## 2. Insert MJP ELI Works + Expressions

In [ ]:
# Settings: choose which MJP hierarchy level becomes one ELI Work/Expression.
INSERT_LEVEL = "actieplan"  # beleidsdoelstelling | actieplan | actie
MJP_SOURCE_FILE = "data/vap_mjp_structured.jsonl"

import json
import math
import re
import uuid
from pathlib import Path

LEVEL_ALIASES = {
    "beleidsdoelstelling": "beleidsdoelstelling",
    "beleidsdoelstellingen": "beleidsdoelstelling",
    "actieplan": "actieplan",
    "actieplannen": "actieplan",
    "actie": "actie",
    "acties": "actie",
    "action": "actie",
    "actions": "actie",
}

LEVEL_LABELS = {
    "beleidsdoelstelling": "Beleidsdoelstelling",
    "actieplan": "Actieplan",
    "actie": "Actie",
}

INSERT_LEVEL = LEVEL_ALIASES[INSERT_LEVEL.strip().lower()]
MJP_NAMESPACE = uuid.NAMESPACE_URL
DUTCH_LANGUAGE_URI = "http://publications.europa.eu/resource/authority/language/NLD"
BODY_URI = "http://data.lblod.info/id/bestuurseenheden/353234a365664e581db5c2f7cc07add2534b47b8e1ab87c821fc6e6365e6bef5"


def is_blank(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    text = str(value).strip()
    return not text or text.lower() == "nan"


def text_value(value) -> str:
    return "" if is_blank(value) else str(value).strip()


def indent(text: str, level: int) -> str:
    return "    " * level + text


def add_optional_line(lines: list[str], label: str, value, level: int) -> None:
    value = text_value(value)
    if value:
        lines.append(indent(f"{label}: {value}", level))


def heading(kind: str, entity: dict, level: int) -> str:
    code_or_id = text_value(entity.get("code")) or f"ID: {text_value(entity.get('id'))}"
    short = text_value(entity.get("korte_omschrijving")) or "zonder korte omschrijving"
    marker = {"beleidsdoelstelling": "====Beleidsdoelstelling", "actieplan": "==Actieplan", "actie": "Actie"}[kind]
    return indent(f"{marker} [{code_or_id}] - {short}", level)


def formatting_actie(act: dict, level: int = 0) -> str:
    lines = [heading("actie", act, level)]
    add_optional_line(lines, "ID", act.get("id"), level + 1)
    add_optional_line(lines, "Lange omschrijving", act.get("lange_omschrijving"), level + 1)
    add_optional_line(lines, "Commentaar", act.get("commentaar"), level + 1)
    return "\n".join(lines)


def formatting_actieplan(ap: dict, level: int = 0) -> str:
    lines = [heading("actieplan", ap, level)]
    add_optional_line(lines, "ID", ap.get("id"), level + 1)
    add_optional_line(lines, "Lange omschrijving", ap.get("lange_omschrijving"), level + 1)
    add_optional_line(lines, "Commentaar", ap.get("commentaar"), level + 1)
    for act in ap.get("acties", []):
        lines.append("")
        lines.append(formatting_actie(act, level + 1))
    return "\n".join(lines)


def formatting_beleidsdoelstelling(bd: dict, level: int = 0) -> str:
    lines = [heading("beleidsdoelstelling", bd, level)]
    add_optional_line(lines, "ID", bd.get("id"), level + 1)
    add_optional_line(lines, "Lange omschrijving", bd.get("lange_omschrijving"), level + 1)
    add_optional_line(lines, "Commentaar", bd.get("commentaar"), level + 1)
    for ap in bd.get("actieplannen", []):
        lines.append("")
        lines.append(formatting_actieplan(ap, level + 1))
    return "\n".join(lines)


def sparql_literal(value, lang: str | None = None, datatype: str | None = None) -> str:
    literal = json.dumps(text_value(value), ensure_ascii=False)
    if lang:
        return f"{literal}@{lang}"
    if datatype:
        return f"{literal}^^{datatype}"
    return literal


def delete_existing_for_level(level: str):
    sparql_update(f"""
{PREFIXES}
DELETE {{
  GRAPH <{PDF_GRAPH}> {{
    ?work ?workPredicate ?workObject .
    ?expression ?expressionPredicate ?expressionObject .
  }}
}}
WHERE {{
  GRAPH <{PDF_GRAPH}> {{
    ?work a eli:Work ;
        mjp:sourceLevel {sparql_literal(level)} ;
        eli:is_realized_by ?expression .
    ?work ?workPredicate ?workObject .
    ?expression ?expressionPredicate ?expressionObject .
  }}
}}
""")


with Path(MJP_SOURCE_FILE).open("r", encoding="utf-8") as f:
    mjps = [json.loads(line) for line in f if line.strip()]

documents = []
for mjp in mjps:
    aanlevering_id = text_value(mjp.get("aanlevering_id"))
    bestuur = text_value(mjp.get("bestuur"))
    year = re.search(r"\d{4}", text_value(mjp.get("rapportjaar")))
    date = f"{year.group(0) if year else '2026'}-01-01"

    for bd in mjp.get("beleidsdoelstellingen", []):
        if INSERT_LEVEL == "beleidsdoelstelling":
            source_id = text_value(bd.get("id"))
            code = text_value(bd.get("code"))
            short = text_value(bd.get("korte_omschrijving")) or "zonder korte omschrijving"
            title = f"{LEVEL_LABELS[INSERT_LEVEL]}{f' {code}' if code else ''} - {short}{f' ({bestuur})' if bestuur else ''}"
            doc_key = f"{aanlevering_id}:{INSERT_LEVEL}:{source_id}"
            documents.append({
                "work_uuid": str(uuid.uuid5(MJP_NAMESPACE, f"mjp-work:{doc_key}")),
                "expression_uuid": str(uuid.uuid5(MJP_NAMESPACE, f"mjp-expression:{doc_key}")),
                "title": title, "date": date, "content": formatting_beleidsdoelstelling(bd),
                "source_id": source_id, "source_code": code, "aanlevering_id": aanlevering_id, "level": INSERT_LEVEL,
            })

        for ap in bd.get("actieplannen", []):
            if INSERT_LEVEL == "actieplan":
                source_id = text_value(ap.get("id"))
                code = text_value(ap.get("code"))
                short = text_value(ap.get("korte_omschrijving")) or "zonder korte omschrijving"
                title = f"{LEVEL_LABELS[INSERT_LEVEL]}{f' {code}' if code else ''} - {short}{f' ({bestuur})' if bestuur else ''}"
                doc_key = f"{aanlevering_id}:{INSERT_LEVEL}:{source_id}"
                documents.append({
                    "work_uuid": str(uuid.uuid5(MJP_NAMESPACE, f"mjp-work:{doc_key}")),
                    "expression_uuid": str(uuid.uuid5(MJP_NAMESPACE, f"mjp-expression:{doc_key}")),
                    "title": title, "date": date, "content": formatting_actieplan(ap),
                    "source_id": source_id, "source_code": code, "aanlevering_id": aanlevering_id, "level": INSERT_LEVEL,
                    "parent_beleidsdoelstelling_id": text_value(bd.get("id")),
                })

            for act in ap.get("acties", []):
                if INSERT_LEVEL == "actie":
                    source_id = text_value(act.get("id"))
                    code = text_value(act.get("code"))
                    short = text_value(act.get("korte_omschrijving")) or "zonder korte omschrijving"
                    title = f"{LEVEL_LABELS[INSERT_LEVEL]}{f' {code}' if code else ''} - {short}{f' ({bestuur})' if bestuur else ''}"
                    doc_key = f"{aanlevering_id}:{INSERT_LEVEL}:{source_id}"
                    documents.append({
                        "work_uuid": str(uuid.uuid5(MJP_NAMESPACE, f"mjp-work:{doc_key}")),
                        "expression_uuid": str(uuid.uuid5(MJP_NAMESPACE, f"mjp-expression:{doc_key}")),
                        "title": title, "date": date, "content": formatting_actie(act),
                        "source_id": source_id, "source_code": code, "aanlevering_id": aanlevering_id, "level": INSERT_LEVEL,
                        "parent_beleidsdoelstelling_id": text_value(bd.get("id")),
                        "parent_actieplan_id": text_value(ap.get("id")),
                    })

print(f"Prepared {len(documents)} MJP {INSERT_LEVEL} documents from {MJP_SOURCE_FILE}")
if documents:
    print("First document preview:")
    print(f"  Title: {documents[0]['title']}")
    print(f"  Work:  http://data.lblod.info/id/works/{documents[0]['work_uuid']}")
    print(f"  Expr:  http://data.lblod.info/id/expressions/{documents[0]['expression_uuid']}")
    print(f"  Text:  {documents[0]['content'][:300].replace(chr(10), ' ')}...")


In [ ]:
CLEAR_EXISTING_FOR_LEVEL = False
DRY_RUN = False  # If True, no changes will be made to the triple store; only print what would be done.
MAX_INSERT = 100  # Limit number of inserts for safety; set to None to insert all prepared documents.

if CLEAR_EXISTING_FOR_LEVEL and not DRY_RUN:
    delete_existing_for_level(INSERT_LEVEL)
    print(f"Deleted previously generated MJP works/expressions for level: {INSERT_LEVEL}")

if DRY_RUN:
    print("DRY_RUN is True; no triples were inserted.")
else:
    for doc in documents[:MAX_INSERT]:  # Limit number of inserts for safety; remove slicing to insert all
        work_uri = f"http://data.lblod.info/id/works/{doc['work_uuid']}"
        expr_uri = f"http://data.lblod.info/id/expressions/{doc['expression_uuid']}"

        opt = (
            (f"\n        mjp:parentBeleidsdoelstellingId {sparql_literal(doc['parent_beleidsdoelstelling_id'])} ;" if doc.get("parent_beleidsdoelstelling_id") else "")
            + (f"\n        mjp:parentActieplanId {sparql_literal(doc['parent_actieplan_id'])} ;" if doc.get("parent_actieplan_id") else "")
        )

        insert_q = f"""
{PREFIXES}
INSERT DATA {{
  GRAPH <{PDF_GRAPH}> {{
    <{work_uri}> a eli:Work ;
        mu:uuid {sparql_literal(doc['work_uuid'])} ;
        dct:title {sparql_literal(doc['title'], lang='nl')} ;
        dct:identifier {sparql_literal(doc['source_id'])} ;
        eli:date_document {sparql_literal(doc['date'], datatype='xsd:date')} ;
        mjp:sourceLevel {sparql_literal(doc['level'])} ;
        mjp:sourceId {sparql_literal(doc['source_id'])} ;
        mjp:sourceCode {sparql_literal(doc['source_code'])} ;
        mjp:aanleveringId {sparql_literal(doc['aanlevering_id'])} ;{opt}
        eli:is_realized_by <{expr_uri}> .

    <{expr_uri}> a eli:Expression ;
        mu:uuid {sparql_literal(doc['expression_uuid'])} ;
        eli:title {sparql_literal(doc['title'], lang='nl')} ;
        eli:language <{DUTCH_LANGUAGE_URI}> ;
        epvoc:expressionContent {sparql_literal(doc['content'], lang='nl')} ;
        <http://mu.semte.ch/vocabularies/ext/owningBody> <{BODY_URI}> ;
        eli:passed_by <{BODY_URI}> .
  }}
}}
"""

        try:
            sparql_update(insert_q)
            print(f"\n--- Successfully inserted MJP {INSERT_LEVEL} ---")
            print(f"Inserted: {doc['title']}")
            print(f"  Work: {work_uri}")
            print(f"  Expr: {expr_uri}")
            print()
        except Exception as e:
            print(f"Failed to insert '{doc['title']}': {e}")

    print(f"--- Inserted {len(documents)} MJP {INSERT_LEVEL} works/expressions ---")


In [ ]:
# URIs
vap_code_list_uri = "http://data.lblod.gift/id/conceptscheme/vap-klimaatadaptatie"
print(vap_code_list_uri)
print(PDF_GRAPH)

## 4. Verify Inserted Data

In [ ]:
verify_query = f"""
{PREFIXES}
SELECT ?level ?sourceId ?work ?title ?expr ?content ?bodyName WHERE {{
  GRAPH <{PDF_GRAPH}> {{
    ?work a eli:Work ;
          dct:title ?title ;
          eli:is_realized_by ?expr .
    ?expr epvoc:expressionContent ?content ;
          eli:passed_by ?body .
    OPTIONAL {{ ?work mjp:sourceLevel ?level . }}
    OPTIONAL {{ ?work mjp:sourceId ?sourceId . }}
    OPTIONAL {{ ?body foaf:name ?bodyName . }}
    FILTER(!BOUND(?level) || ?level = "{INSERT_LEVEL}")
    FILTER(LANG(?title) = "nl" || LANG(?title) = "en" || LANG(?title) = "")
  }}
}}
ORDER BY ?title
LIMIT 1000
"""


results = sparql_query(verify_query)
print(f"Found {len(results)} ELI Works with Expressions:\n")
for r in results:
    level = r.get("level", {}).get("value", "N/A")
    source_id = r.get("sourceId", {}).get("value", "N/A")
    title = r.get("title", {}).get("value", "N/A")
    work = r.get("work", {}).get("value", "N/A")
    expr = r.get("expr", {}).get("value", "N/A")
    body_name = r.get("bodyName", {}).get("value", "N/A")
    content = r.get("content", {}).get("value", "N/A")[:160].replace("\n", " ")
    print(f"  [{level} {source_id}] {title}")
    print(f"  Work:  {work}")
    print(f"  Expr:  {expr}")
    print(f"  Body:  {body_name}")
    print(f"  Text:  {content}...")
    print()

In [ ]:
print(len(results), "documents found in triple store with ELI Work and Expression.")
for r in results[50:]:
    expr = r.get("expr", {}).get("value", "N/A")
    print(expr)
    

## 5. Verify Codelist Data

In [ ]:
codelist_verify = f"""
{PREFIXES}
SELECT ?scheme ?concept ?label WHERE {{
  GRAPH <{PUBLIC_GRAPH}> {{
    ?concept a skos:Concept ;
        skos:inScheme ?scheme ;
        skos:prefLabel ?label .
    FILTER(LANG(?label) = "en" || LANG(?label) = "nl" || LANG(?label) = "")
  }}
}}
ORDER BY ?label
"""

results = sparql_query(codelist_verify)
print(f"Found {len(results)} concepts in codelists:\n")
for r in results:
    scheme = r.get("scheme", {}).get("value", "N/A").split("/")[-1]
    label = r.get("label", {}).get("value", "N/A")
    concept = r.get("concept", {}).get("value", "N/A")
    print(f"  [{scheme}] {label}")
    print(f"    URI: {concept}")

## 6. Check for Annotations (after running codelist mapping job)

Run this cell AFTER creating and running a codelist mapping job from the dashboard.

In [ ]:
ann_query = f"""
{PREFIXES}
SELECT ?annotation ?target ?body ?bodyLabel ?motivation WHERE {{
  ?annotation a oa:Annotation ;
      oa:hasTarget ?target ;
      oa:hasBody ?body .
  OPTIONAL {{ ?annotation oa:motivatedBy ?motivation . }}
  OPTIONAL {{
    ?body skos:prefLabel ?bodyLabel .
    FILTER(LANG(?bodyLabel) = "en" || LANG(?bodyLabel) = "")
  }}
}}
ORDER BY ?target
"""

results = sparql_query(ann_query)
print(f"Found {len(results)} annotations:\n")
for r in results:
    target = r.get("target", {}).get("value", "N/A").split("/")[-1]
    body_label = r.get("bodyLabel", {}).get("value", r.get("body", {}).get("value", "N/A"))
    motivation = r.get("motivation", {}).get("value", "N/A").split("/")[-1]
    print(f"  Target: ...{target}")
    print(f"  Body:   {body_label}")
    print(f"  Motiv:  {motivation}")
    print()

## 7. Cleanup (Optional)

Remove the test data inserted by this notebook.

In [ ]:
# WARNING: This removes generated MJP works/expressions for the configured INSERT_LEVEL.
# It leaves codelists and generated bestuur organization labels in place.

cleanup_mjp_level = f"""
{PREFIXES}
DELETE {{
  GRAPH <{PDF_GRAPH}> {{
    ?work ?workPredicate ?workObject .
    ?expression ?expressionPredicate ?expressionObject .
  }}
}}
WHERE {{
  GRAPH <{PDF_GRAPH}> {{
    ?work a eli:Work ;
        mjp:sourceLevel "{INSERT_LEVEL}" ;
        eli:is_realized_by ?expression .
    ?work ?workPredicate ?workObject .
    ?expression ?expressionPredicate ?expressionObject .
  }}
}}
"""

# Uncomment to execute:
# sparql_update(cleanup_mjp_level)
# print(f"Removed generated MJP works/expressions for level: {INSERT_LEVEL}")
print("Cleanup cell is commented out. Uncomment to execute.")